# __Master Pipeline__

# **0. Librairies**

In [1]:
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
import pandas as pd
import sys
import os


sys.path.append(os.path.abspath(".."))

from lib.data.dataset import BeeDataset
from lib.data.preprocessing import TorchPreprocessor
from lib.data.train_val_split import train_val_split
from lib.data.preprocessing import TorchPreprocessor
from lib.data.data_augmentation import data_augmented_loader
from lib.utils.model_saver import ModelSaver

In [ ]:
## CONFIGURATION
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 32
NUM_EPOCHS = 25
LR = 1e-4

DROPOUT = 0.4
WEIGHT_DECAY = 1e-4

NUM_CLASSES = 50

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

## **1. Preprocessing**

In [3]:
train_loader, val_loader = data_augmented_loader(IMAGENET_MEAN, IMAGENET_STD, target_size=(224, 224))

Train prêt : 6417 images (avec augmentation ciblée)
Val prête  : 1582 images (sans augmentation)


In [4]:
# --- Récupérer tous les labels du val_loader ---
all_labels = []

for _, labels in val_loader:
    all_labels.extend(labels.numpy() if isinstance(labels, torch.Tensor) else labels)

# --- Compter le nombre d'images par classe ---
class_counts_val = pd.Series(all_labels).value_counts().sort_index()

# --- Créer un DataFrame ---
df_val_counts = pd.DataFrame({
    "class": class_counts_val.index,
    "num_images": class_counts_val.values
})

# --- Sauvegarder dans un CSV ---
csv_val_path = "val_class_counts.csv"
df_val_counts.to_csv(csv_val_path, index=False)
print(f"CSV des images par classe dans la validation sauvegardé dans : {csv_val_path}")

CSV des images par classe dans la validation sauvegardé dans : val_class_counts.csv


## **2. Modèle**

In [5]:
val_class_counts = pd.read_csv("val_class_counts.csv")
weights = 1.0 / val_class_counts["num_images"].values

In [6]:
class ResnetFineTune (nn.Module):
    def __init__(self, num_classes, 
                 dropout, 
                 freeze = False):
        
        super(ResnetFineTune, self).__init__()

        self.resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        
        if freeze:
            for param in self.resnet.parameters():
                param.requires_grad = False

            for param in self.resnet.layer4.parameters():
                param.requires_grad = True

        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, num_classes),
        )

    def forward (self, x):
        return self.resnet(x)
    
    def inference(self, x):
        self.eval()
        with torch.no_grad():
            logits = self(x)
            probs = torch.softmax(logits, dim=1)
            pred_class = torch.argmax(probs, dim=1).item()
        
        return pred_class

In [7]:
model = ResnetFineTune(NUM_CLASSES, DROPOUT)
model.to(DEVICE)

ResnetFineTune(
  (resnet): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, tra

In [8]:
criterion = nn.CrossEntropyLoss()

In [9]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

In [ ]:
from datetime import datetime
import os
import json
import torch
from sklearn.metrics import confusion_matrix
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
import pandas as pd
import numpy as np


from lib.data.dataset import BeeDataset
from torch.utils.data import DataLoader
import pandas as pd


class ModelSaver:
    def __init__(self, model, username="bizuuuuth"):
        self.model = model
        self.folder_path = self.create_model_folder(model, username)


    def save_training_config(self, model, optimizer, BATCH_SIZE, NUM_EPOCHS, LR, DEVICE, scheduler=None, criterion=None):
        summary = {}

        # ===== MODEL =====
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

        summary["model"] = {
            "architecture": type(model).__name__,
            "total_params": total_params,
            "trainable_params": trainable_params,
        }

        # ===== TRAINING CONFIG =====
        summary["training_config"] = {
            "batch_size": BATCH_SIZE,
            "num_epochs": NUM_EPOCHS,
            "learning_rate": LR,
            "device": str(DEVICE),
            "dropout": getattr(model, "dropout", "N/A"),  # Si le modèle a un attribut dropout, on le récupère
        }

        # ===== OPTIMIZER =====
        first_group = optimizer.param_groups[0]

        optim_keys = ["lr", "weight_decay", "momentum", "betas", "eps"]
        optimizer_info = {
            "type": type(optimizer).__name__,
        }

        for key in optim_keys:
            if key in first_group:
                optimizer_info[key] = first_group[key]

        summary["optimizer"] = optimizer_info

        # ===== SCHEDULER =====
        if scheduler is not None:
            scheduler_info = {
                "type": type(scheduler).__name__,
            }

            # Récupération simple des attributs principaux
            possible_attrs = [
                "T_max",
                "eta_min",
                "gamma",
                "step_size",
                "patience",
                "factor",
                "total_steps",
                "base_lrs",
            ]

            for attr in possible_attrs:
                if hasattr(scheduler, attr):
                    scheduler_info[attr] = getattr(scheduler, attr)

            summary["scheduler"] = scheduler_info
        else:
            summary["scheduler"] = "None (Constant LR)"

        # ===== CRITERION =====
        if criterion is not None:
            criterion_info = {
                "type": type(criterion).__name__,
            }

            if hasattr(criterion, "label_smoothing"):
                criterion_info["label_smoothing"] = criterion.label_smoothing

            if hasattr(criterion, "weight") and criterion.weight is not None:
                criterion_info["class_weights"] = True

            summary["criterion"] = criterion_info

        json_path = os.path.join(self.folder_path, "training_summary.json")
        with open(json_path, "w") as f:
            json.dump(summary, f, indent=4)
        

    def create_model_folder(self, model, username="bizuuuuth"):
        model_name = type(model).__name__
        hour_min = datetime.now().strftime("%Hh%M")
        folder_name = f"{model_name}_{username}_{hour_min}"
        folder_path = os.path.join("../results", folder_name)
        os.makedirs(folder_path, exist_ok=True)
        return folder_path
    
    def load_model(self, model, model_name="best_model.pth", device="cpu"):
        model_path = os.path.join(self.folder_path, model_name)
        model.load_state_dict(torch.load(model_path, map_location=torch.device(device)))
        model.to(device)
        model.eval()
        return model
    
    def submission(self, model, batch_size=32, transform=None, 
               model_name="best_model.pth", device="cpu"):
    
        model = self.load_model(model, model_name, device)

        test_dataset = BeeDataset(train=False, transform=transform)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

        all_ids, all_preds = [], []

        with torch.no_grad():
            for imgs, ids in test_loader:
                imgs = imgs.to(device)
                outputs = model(imgs)
                preds = torch.argmax(outputs, dim=1)

                all_preds.extend(preds.cpu().tolist())
                all_ids.extend(
                    [int(x) if isinstance(x, torch.Tensor) else x for x in ids]
                )

        submission_df = pd.DataFrame({
            "id": all_ids,
            "label": all_preds
        })

        save_path = os.path.join(self.folder_path, "submission.csv")
        submission_df.to_csv(save_path, index=False)

        print(f"Submission saved to {save_path}")

    

    def evaluate(self, model, batch_size=32, transform=None,
                model_name="best_model.pth", device="cpu",
                num_classes=None):

        model = self.load_model(model, model_name, device)

        val_dataset = BeeDataset(train=False, transform=transform)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        all_preds, all_labels = [], []

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                preds = torch.argmax(outputs, dim=1)

                all_preds.extend(preds.cpu().tolist())
                all_labels.extend(labels.cpu().tolist())

        if num_classes is None:
            num_classes = max(max(all_labels), max(all_preds)) + 1

        cm = confusion_matrix(all_labels, all_preds)
        cm_df = pd.DataFrame(cm)

        cm_path = os.path.join(self.folder_path, "confusion_matrix.csv")
        cm_df.to_csv(cm_path, index=False)

        # ===== Metrics =====
        metrics = []

        total = cm.sum()

        for cls in range(num_classes):
            TP = cm[cls, cls]
            FP = cm[:, cls].sum() - TP
            FN = cm[cls, :].sum() - TP
            TN = total - (TP + FP + FN)

            precision = TP / (TP + FP) if (TP + FP) > 0 else 0
            recall = TP / (TP + FN) if (TP + FN) > 0 else 0
            f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            accuracy_cls = (TP + TN) / total

            metrics.append({
                "class": cls,
                "accuracy": accuracy_cls,
                "precision": precision,
                "recall": recall,
                "f1_score": f1
            })

        metrics_df = pd.DataFrame(metrics)

        metrics_path = os.path.join(self.folder_path, "metrics_per_class.csv")
        metrics_df.to_csv(metrics_path, index=False)

        print(f"Confusion matrix saved to {cm_path}")
        print(f"Metrics saved to {metrics_path}")

        return cm_df, metrics_df

        return cm, metrics_df

    def save_model(self, model, name="best_model.pth"):
        path = os.path.join(self.folder_path, name)
        torch.save(model.state_dict(), path)
        return path


    def save_confusion_matrix(self, cm, name="confusion_matrix.csv"):
        path = os.path.join(self.folder_path, name)
        pd.DataFrame(cm).to_csv(path, index=False)


    def save_metrics(self, metrics_df, name="metrics_per_class.csv"):
        path = os.path.join(self.folder_path, name)
        metrics_df.to_csv(path, index=False)


    def save_training_log(self, log_row, fieldnames, name="training_log.csv"):
        path = os.path.join(self.folder_path, name)

        file_exists = os.path.isfile(path)

        with open(path, mode='a', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            if not file_exists:
                writer.writeheader()
            writer.writerow(log_row)
    
    def save_epoch(self, epoch, metrics, mode="train"):
        training_monitor_path = os.path.join(self.folder_path, "training_monitor.json")
        
        f1_per_class = metrics["f1_per_class"]

        epoch_row = {   
            "mode": mode,
            "f1_macro": float(metrics["f1_macro"]), # Sécurité pour JSON
            "f1_per_class": f1_per_class,
            "accuracy": float(metrics.get("accuracy", 0)),
            "precision_per_class": metrics.get("precision_per_class", []),
            "recall_per_class": metrics.get("recall_per_class", [])
        }

        if not os.path.isfile(training_monitor_path):
            data = {}
        else:
            with open(training_monitor_path, mode='r', encoding='utf-8') as f:
                try:
                    data = json.load(f)
                except json.JSONDecodeError:
                    data = {}

        epoch_key = str(epoch)
        if epoch_key not in data:
            data[epoch_key] = []
        
        data[epoch_key].append(epoch_row)

        with open(training_monitor_path, mode='w', encoding='utf-8') as f:
            json.dump(data, f, indent=4)

In [11]:
ModelSaver = ModelSaver(model=model, username="temp")
ModelSaver.save_training_config(model, optimizer, BATCH_SIZE, NUM_EPOCHS, LR, DEVICE, criterion=criterion)

## **3. F1-score**

In [12]:
import numpy as np

def compute_f1(all_labels, all_preds, num_classes):
    f1_per_class = []

    for cls in range(num_classes):
        # True Positives
        TP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) == cls))
        # False Positives
        FP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) != cls))
        # False Negatives
        FN = np.sum((np.array(all_preds) != cls) & (np.array(all_labels) == cls))

        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0

        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_per_class.append(f1)

    # F1 macro : moyenne des classes
    f1_macro = np.mean(f1_per_class)
    return f1_macro, f1_per_class

In [ ]:
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

def compute_all_metrics(all_labels, all_preds, num_classes):
    """
    Calcule toutes les métriques en utilisant ta fonction compute_f1 personnalisée.
    """
    all_preds_np = np.array(all_preds)
    all_labels_np = np.array(all_labels)
    acc = np.mean(all_preds_np == all_labels_np)

    f1_macro, f1_per_class = compute_f1(all_labels, all_preds, num_classes)

    precision_per_class = precision_score(all_labels, all_preds, average=None, labels=range(num_classes), zero_division=0)
    recall_per_class = recall_score(all_labels, all_preds, average=None, labels=range(num_classes), zero_division=0)

    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_per_class": f1_per_class,
        "precision_per_class": precision_per_class,
        "recall_per_class": recall_per_class
    }

In [ ]:
def compute_confusion_matrix(model, data_loader, device="cpu", num_classes=None):
    model.to(device)
    model.eval()

    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in data_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    if num_classes is None:
        num_classes = max(max(all_labels), max(all_preds)) + 1

    cm = confusion_matrix(all_labels, all_preds)
    return cm

## **4. Fonctions de training et validation**

In [14]:
def train_one_epoch(epoch):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    for images, labels in tqdm(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    # 🔹 Calcul F1 avec ta fonction existante
    all_metrics = compute_all_metrics(all_labels, all_preds, NUM_CLASSES)
    ModelSaver.save_epoch(epoch, all_metrics, mode="train")

    f1_macro = all_metrics["f1_macro"]
    f1_per_class = all_metrics["f1_per_class"]

    # 🔹 Affichage
    print(f"Train F1 macro: {f1_macro:.4f}")

    return total_loss / len(train_loader), correct / total, f1_macro, f1_per_class


In [15]:
import torch

def validate(epoch):
    model.eval()
    total_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calcul F1 manuel par classe
    all_metrics = compute_all_metrics(all_labels, all_preds, NUM_CLASSES)
    ModelSaver.save_epoch(epoch, all_metrics, mode="val")

    f1_macro = all_metrics["f1_macro"]
    f1_per_class = all_metrics["f1_per_class"]

    return (total_loss / len(val_loader), f1_macro, f1_per_class, np.array(all_preds), np.array(all_labels))

## **5. Entrainement**

**Entrainement**

In [ ]:
import csv
import pandas as pd
from sklearn.metrics import confusion_matrix

best_f1 = 0.0

csv_file = "training_log.csv"
fieldnames = ['epoch', 'train_loss', 'train_acc', 'train_f1_macro',
              'val_loss', 'val_f1_macro']


loss_train, loss_val = [], []

for epoch in range(NUM_EPOCHS):

    # ===== TRAIN =====
    train_loss, train_acc, train_f1_macro, train_f1_per_class = train_one_epoch(epoch)
    loss_train.append(train_loss)
    # scheduler.step()

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Train F1 Macro: {train_f1_macro:.4f}")

    # ===== VALIDATION =====
    val_loss, val_f1_macro, val_f1_per_class, val_preds, val_labels = validate(epoch)
    loss_val.append(val_loss)
    
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val F1 Macro: {val_f1_macro:.4f}")



    # ===== BEST MODEL =====
    if val_f1_macro > best_f1:
        best_f1 = val_f1_macro

        ModelSaver.save_model(model, name=f"best_model_epoch_{epoch}.pth")

        cm = compute_confusion_matrix(model, val_loader, device=DEVICE, num_classes=NUM_CLASSES)
        ModelSaver.save_confusion_matrix(cm, name=f"confusion_matrix_epoch_{epoch}.csv")        

        print(f" Nouveau meilleur modèle sauvegardé ! F1 Macro = {best_f1:.4f}")


       

  0%|          | 0/201 [00:03<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 3.95 GiB of which 10.12 MiB is free. Process 39829 has 3.23 GiB memory in use. Including non-PyTorch memory, this process has 718.00 MiB memory in use. Of the allocated memory 613.42 MiB is allocated by PyTorch, and 50.58 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
plt.plot(loss_train, 'r' , marker='o', linestyle='-', label = 'loss_train')
plt.plot(loss_val, 'b' , marker='o', linestyle='-', label = 'loss_test')
plt.xlabel('Epoch')
plt.ylabel('Loss fonction')
plt.grid()
plt.legend()
plt.show()